In [1]:
p_h, p_d = 0.60, 0.40
p_a_h, p_a_d = 0.30, 0.20

p_h_a = p_h * p_a_h / (p_h * p_a_h + p_d * p_a_d)
print("P(Hosteler | A Grade) =", round(p_h_a, 4))

p_disease, p_no = 0.01, 0.99
p_pos_disease, p_pos_no = 0.99, 0.02
p_disease_pos = p_disease * p_pos_disease / (p_disease * p_pos_disease + p_no * p_pos_no)

print("P(Disease | Positive) =", round(p_disease_pos, 4))

P(Hosteler | A Grade) = 0.6923
P(Disease | Positive) = 0.3333


In [2]:
import pandas as pd
from collections import defaultdict

df = pd.read_csv("buyers.csv")

def train_nb(df, target):
    classes = df[target].unique()
    model = {"prior": {}, "cond": defaultdict(dict)}
    for c in classes:
        d = df[df[target] == c]
        model["prior"][c] = len(d) / len(df)
        for col in df.columns[:-1]:
            for v in df[col].unique():
                model["cond"][col, v, c] = (sum(d[col] == v) + 1) / (len(d) + len(df[col].unique()))
    return model

def predict(model, x):
    scores = {}
    for c in model["prior"]:
        p = model["prior"][c]
        for col, v in x.items():
            p *= model["cond"][col, v, c]
        scores[c] = p
    return max(scores, key=scores.get), scores

model = train_nb(df, "buys_computer")
x = {"age": "<=30", "income": "medium", "student": "yes", "credit_rating": "fair"}
print("Prediction:", predict(model, x)[0])

Prediction: yes


In [3]:
import re
from collections import Counter

df = pd.read_csv("sports.csv")
def train_text_nb(df):
    model = {}
    for c in df["Tag"].unique():
        words = Counter(" ".join(df[df.Tag == c]["Text"]).lower().split())
        model[c] = {
            "prior": sum(df.Tag == c) / len(df),
            "words": words,
            "total": sum(words.values())}
    return model

def predict_text(model, text):
    words = re.findall(r'\b\w+\b', text.lower())
    scores = {}
    for c, m in model.items():
        p = m["prior"]
        for w in words:
            p *= (m["words"][w] + 1) / (m["total"] + len(m["words"]))
        scores[c] = p
    return max(scores, key=scores.get)

model = train_text_nb(df)
print("Prediction:", predict_text(model, "A very close game"))

Prediction: Sports
